# Storage Inventory Calculation from Transaction Movements

This notebook processes transaction data to calculate the current inventory (products and quantities) for each emplacement based on movement history.

## Logic:
- **Source movements** (id_emplacement_source): Subtract quantity from that emplacement (outgoing)
- **Destination movements** (id_emplacement_destination): Add quantity to that emplacement (incoming)
- **Exclusions**: 'PRODUCTION LOCATION' as source and 'CUSTOMER LOCATION' as destination are filtered out
- **Result**: JSON with each emplacement's products and positive quantities only

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import defaultdict

# Setup paths
DATA_PATH = Path('../../data/raw')
OUTPUT_PATH = Path('../../data/processed')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("Environment setup complete!")

Environment setup complete!


## 1. Load Data

Load the three main tables: emplacements, lignes_transaction, and produits.

In [3]:
# Load data files
try:
    emplacements_df = pd.read_csv('emplacements.csv')
    lignes_transaction_df = pd.read_csv('lignes_transaction.csv')
    produits_df = pd.read_csv('produits.csv')
    
    print(f"✓ Emplacements loaded: {len(emplacements_df)} rows")
    print(f"✓ Lignes transaction loaded: {len(lignes_transaction_df)} rows")
    print(f"✓ Produits loaded: {len(produits_df)} rows")
    
except FileNotFoundError as e:
    print(f"❌ Error loading files: {e}")
    print("\nPlease ensure the following CSV files exist in the data/raw directory:")
    print("  - emplacements.csv")
    print("  - lignes_transaction.csv")
    print("  - produits.csv")

✓ Emplacements loaded: 840 rows
✓ Lignes transaction loaded: 76890 rows
✓ Produits loaded: 1583 rows


C:\Users\GAMER\AppData\Local\Temp\ipykernel_19816\967212817.py:4: DtypeWarning: Columns (1,2,3,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  lignes_transaction_df = pd.read_csv('lignes_transaction.csv')


In [4]:
# Inspect the data structure
print("=== EMPLACEMENTS ===")
print(emplacements_df.head())
print(f"\nColumns: {list(emplacements_df.columns)}")
print(f"\n=== LIGNES TRANSACTION (first 5) ===")
print(lignes_transaction_df.head())
print(f"\nColumns: {list(lignes_transaction_df.columns)}")
print(f"\n=== PRODUITS ===")
print(produits_df.head())
print(f"\nColumns: {list(produits_df.columns)}")

=== EMPLACEMENTS ===
   id_emplacement code_emplacement  id_entrepot    zone type_emplacement  \
0             418         0A-01-01          235  B7-PCK          PICKING   
1             422         0A-01-02          235  B7-PCK          PICKING   
2             426         0A-01-03          235  B7-PCK          PICKING   
3             430         0A-02-01          235  B7-PCK          PICKING   
4             434         0A-02-02          235  B7-PCK          PICKING   

   actif  Volume (m3)  Unnamed: 7  
0   True          NaN         NaN  
1   True          NaN         NaN  
2   True          NaN         NaN  
3   True          NaN         NaN  
4   True          NaN         NaN  

Columns: ['id_emplacement', 'code_emplacement', 'id_entrepot', 'zone', 'type_emplacement', 'actif', 'Volume (m3)', 'Unnamed: 7']

=== LIGNES TRANSACTION (first 5) ===
  id_transaction no_ligne id_produit quantite id_emplacement_source  \
0          texte   entier      texte   nombre                 texte

## 2. Process Transaction Movements

Calculate inventory for each emplacement by:
1. Processing source movements (products leaving - negative impact)
2. Processing destination movements (products arriving - positive impact)
3. Filtering out special locations ('PRODUCTION LOCATION', 'CUSTOMER LOCATION')
4. Keeping only positive final quantities

In [31]:
def is_numeric_emplacement(val):
    """Check if value is a number (can be converted to int)"""
    if pd.isna(val):
        return False
    try:
        int(val)
        return True
    except (ValueError, TypeError):
        return False

def calculate_emplacement_inventory(lignes_df):
    """
    Calculate inventory for each emplacement based on transaction movements.
    
    Returns:
        dict: {emplacement_id: {product_id: quantity}}
    """
    # Dictionary to store inventory: {emplacement_id: {product_id: quantity}}
    inventory = defaultdict(lambda: defaultdict(int))
    
    processed_source = 0
    processed_dest = 0
    skipped_source = 0
    skipped_dest = 0
    skipped_invalid = 0
    
    print("Processing transaction movements...")
    print("\n=== SAMPLE TRANSACTION LOGS (first 10) ===")
    log_count = 0
    
    for idx, row in lignes_df.iterrows():
        # Convert quantite to numeric, skip if invalid
        quantite = pd.to_numeric(row['quantite'], errors='coerce')
        if pd.isna(quantite):
            skipped_invalid += 1
            continue
            
        id_produit = row['id_produit']
        id_emplacement_source = row['id_emplacement_source']
        id_emplacement_destination = row['id_emplacement_destination']
        
        # Process SOURCE (outgoing movement - negative quantity)
        if pd.notna(id_emplacement_source) and id_emplacement_source != 'PRODUCTION LOCATION':
            if is_numeric_emplacement(id_emplacement_source):
                emplacement_id = str(int(id_emplacement_source))
                inventory[emplacement_id][id_produit] -= quantite
                
                # Log first 10 source movements
                if log_count < 10:
                    print(f"  SOURCE: Emplacement {emplacement_id} | Product {id_produit} | -{quantite} (subtract)")
                    log_count += 1
                
                processed_source += 1
            else:
                skipped_source += 1
        
        # Process DESTINATION (incoming movement - positive quantity)
        if pd.notna(id_emplacement_destination) and id_emplacement_destination != 'CUSTOMER LOCATION':
            if is_numeric_emplacement(id_emplacement_destination):
                emplacement_id = str(int(id_emplacement_destination))
                inventory[emplacement_id][id_produit] += quantite
                
                # Log first 10 destination movements
                if log_count < 50:
                    print(f"  DEST:   Emplacement {emplacement_id} | Product {id_produit} | +{quantite} (add)")
                    log_count += 1
                
                processed_dest += 1
            else:
                skipped_dest += 1
    
    print(f"\n✓ Processed {processed_source} source movements")
    print(f"✓ Processed {processed_dest} destination movements")
    print(f"  Skipped {skipped_invalid} rows with invalid quantity")
    print(f"  Skipped {skipped_source} non-numeric sources")
    print(f"  Skipped {skipped_dest} non-numeric destinations")
    
    return inventory

# Calculate inventory
raw_inventory = calculate_emplacement_inventory(lignes_transaction_df)
print(f"\n✓ Calculated inventory for {len(raw_inventory)} emplacements")

Processing transaction movements...

=== SAMPLE TRANSACTION LOGS (first 10) ===
  SOURCE: Emplacement 54 | Product 31851 | -80 (subtract)
  SOURCE: Emplacement 609 | Product 31501 | -120 (subtract)
  SOURCE: Emplacement 610 | Product 31501 | -480 (subtract)
  SOURCE: Emplacement 35 | Product 31490 | -160 (subtract)
  SOURCE: Emplacement 39 | Product 31496 | -40 (subtract)
  SOURCE: Emplacement 531 | Product 31474 | -100 (subtract)
  SOURCE: Emplacement 366 | Product 32089 | -30 (subtract)
  SOURCE: Emplacement 628 | Product 32085 | -30 (subtract)
  SOURCE: Emplacement 628 | Product 32079 | -30 (subtract)
  SOURCE: Emplacement 303 | Product 32124 | -90 (subtract)
  DEST:   Emplacement 628 | Product 38744 | +822 (add)
  DEST:   Emplacement 549 | Product 31591 | +1000 (add)
  DEST:   Emplacement 364 | Product 31565 | +39800 (add)
  DEST:   Emplacement 625 | Product 34015 | +7500 (add)
  DEST:   Emplacement 32 | Product 31847 | +300 (add)
  DEST:   Emplacement 215 | Product 38720 | +2000 (

## 3. Clean Inventory (Keep Only Positive Quantities)

In [8]:
def clean_inventory(raw_inventory):
    """
    Keep only positive quantities in inventory.
    
    Args:
        raw_inventory: dict with emplacement_id -> {product_id: quantity}
    
    Returns:
        dict: Cleaned inventory with only positive quantities
    """
    clean_inv = {}
    
    total_products_before = 0
    total_products_after = 0
    negative_removed = 0
    
    for emplacement_id, products in raw_inventory.items():
        clean_inv[emplacement_id] = {}
        
        for product_id, quantity in products.items():
            total_products_before += 1
            
            if quantity > 0:
                clean_inv[emplacement_id][product_id] = quantity
                total_products_after += 1
            else:
                negative_removed += 1
        
        # Remove emplacements with no products
        if not clean_inv[emplacement_id]:
            del clean_inv[emplacement_id]
    
    print(f"Cleaning inventory...")
    print(f"  Total product entries before: {total_products_before}")
    print(f"  Removed negative/zero quantities: {negative_removed}")
    print(f"  Final product entries: {total_products_after}")
    print(f"  Emplacements with inventory: {len(clean_inv)}")
    
    return clean_inv

# Clean the inventory
final_inventory = clean_inventory(raw_inventory)
print(f"\n✓ Final inventory ready: {len(final_inventory)} emplacements")

Cleaning inventory...
  Total product entries before: 6041
  Removed negative/zero quantities: 3593
  Final product entries: 2448
  Emplacements with inventory: 658

✓ Final inventory ready: 658 emplacements


## 4. Inspect Results

In [9]:
# Display sample results
print("=== SAMPLE INVENTORY DATA ===\n")

# Show first 5 emplacements with their products
sample_count = 0
for emplacement_id, products in sorted(final_inventory.items())[:5]:
    print(f"Emplacement ID: {emplacement_id}")
    print(f"  Products: {len(products)}")
    
    # Show first 3 products for this emplacement
    for i, (product_id, quantity) in enumerate(list(products.items())[:3]):
        print(f"    - Product {product_id}: {quantity} units")
    
    if len(products) > 3:
        print(f"    ... and {len(products) - 3} more products")
    print()
    
    sample_count += 1

# Statistics
total_products_stored = sum(len(products) for products in final_inventory.values())
print(f"\n=== STATISTICS ===")
print(f"Total emplacements with inventory: {len(final_inventory)}")
print(f"Total unique product-location pairs: {total_products_stored}")
print(f"Average products per emplacement: {total_products_stored / len(final_inventory):.2f}")

=== SAMPLE INVENTORY DATA ===

Emplacement ID: 1
  Products: 5
    - Product 34410: 1000 units
    - Product 35582: 950 units
    - Product 31943: 80 units
    ... and 2 more products

Emplacement ID: 1002
  Products: 1
    - Product 31509: 2640 units

Emplacement ID: 1003
  Products: 1
    - Product 31960: 1020 units

Emplacement ID: 1006
  Products: 1
    - Product 31951: 3840 units

Emplacement ID: 1011
  Products: 1
    - Product 31511: 1275 units


=== STATISTICS ===
Total emplacements with inventory: 658
Total unique product-location pairs: 2448
Average products per emplacement: 3.72


## 5. Export to JSON

Save the inventory data in JSON format for easy access and integration.

In [10]:
# Convert to JSON-serializable format
# Convert product IDs to strings if they're numeric
json_inventory = {}
for emplacement_id, products in final_inventory.items():
    json_inventory[emplacement_id] = {str(prod_id): qty for prod_id, qty in products.items()}

# Save to JSON file
output_file = OUTPUT_PATH / 'emplacement_inventory.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(json_inventory, f, indent=2, ensure_ascii=False)

print(f"✓ Inventory exported to: {output_file}")
print(f"  File size: {output_file.stat().st_size / 1024:.2f} KB")

✓ Inventory exported to: ..\..\data\processed\emplacement_inventory.json
  File size: 58.45 KB


In [11]:
# Display a sample of the JSON structure
print("=== JSON STRUCTURE EXAMPLE ===\n")
sample_emplacement = list(json_inventory.keys())[0] if json_inventory else None

if sample_emplacement:
    print(f'"{sample_emplacement}": {{')
    for i, (prod_id, qty) in enumerate(list(json_inventory[sample_emplacement].items())[:3]):
        comma = "," if i < min(2, len(json_inventory[sample_emplacement]) - 1) else ""
        print(f'  "{prod_id}": {qty}{comma}')
    if len(json_inventory[sample_emplacement]) > 3:
        print(f'  ... ({len(json_inventory[sample_emplacement]) - 3} more products)')
    print("}")
else:
    print("No inventory data to display")

print("\n=== PROCESS COMPLETE ===")
print(f"✓ Inventory JSON saved to: {output_file.name}")
print(f"✓ Ready for use in optimization algorithms!")

=== JSON STRUCTURE EXAMPLE ===

"54": {
  "31850": 1120
}

=== PROCESS COMPLETE ===
✓ Inventory JSON saved to: emplacement_inventory.json
✓ Ready for use in optimization algorithms!


## 6. (Optional) Create Enriched Inventory with Product Details

Optionally, create a more detailed version that includes product information (SKU, name, etc.)

In [12]:
# Create enriched inventory with product details
enriched_inventory = {}

# Create a product lookup dictionary
product_lookup = produits_df.set_index('id_produit').to_dict('index')

for emplacement_id, products in final_inventory.items():
    enriched_inventory[emplacement_id] = []
    
    for product_id, quantity in products.items():
        product_info = product_lookup.get(product_id, {})
        
        enriched_inventory[emplacement_id].append({
            'id_produit': int(product_id) if str(product_id).isdigit() else product_id,
            'sku': product_info.get('sku', 'N/A'),
            'nom_produit': product_info.get('nom_produit', 'Unknown'),
            'categorie': product_info.get('categorie', 'N/A'),
            'quantite': quantity,
            'volume_total_m3': quantity * product_info.get('volume pcs (m3)', 0) if 'volume pcs (m3)' in product_info else None,
            'poids_total_kg': quantity * product_info.get('Poids(kg)', 0) if 'Poids(kg)' in product_info else None
        })

# Save enriched inventory
enriched_output_file = OUTPUT_PATH / 'emplacement_inventory_enriched.json'
with open(enriched_output_file, 'w', encoding='utf-8') as f:
    json.dump(enriched_inventory, f, indent=2, ensure_ascii=False)

print(f"✓ Enriched inventory exported to: {enriched_output_file}")
print(f"  File size: {enriched_output_file.stat().st_size / 1024:.2f} KB")
print(f"\nEnriched format includes: SKU, product name, category, volume, and weight")

✓ Enriched inventory exported to: ..\..\data\processed\emplacement_inventory_enriched.json
  File size: 515.06 KB

Enriched format includes: SKU, product name, category, volume, and weight


In [13]:
# Display a sample of enriched inventory
print("=== ENRICHED INVENTORY SAMPLE ===\n")

if enriched_inventory:
    sample_emp_id = list(enriched_inventory.keys())[0]
    print(f"Emplacement ID: {sample_emp_id}")
    print(f"Products stored: {len(enriched_inventory[sample_emp_id])}\n")
    
    # Show first 2 products with full details
    for product in enriched_inventory[sample_emp_id][:2]:
        print(f"  Product: {product['nom_produit']}")
        print(f"    SKU: {product['sku']}")
        print(f"    Category: {product['categorie']}")
        print(f"    Quantity: {product['quantite']} units")
        if product['volume_total_m3']:
            print(f"    Total Volume: {product['volume_total_m3']:.3f} m³")
        if product['poids_total_kg']:
            print(f"    Total Weight: {product['poids_total_kg']:.2f} kg")
        print()
else:
    print("No enriched inventory data to display")

print("✅ All processing complete!")

=== ENRICHED INVENTORY SAMPLE ===

Emplacement ID: 54
Products stored: 1

  Product: Unknown
    SKU: N/A
    Category: N/A
    Quantity: 1120 units

✅ All processing complete!


## 7. Calculate Total Volume per Emplacement

Calculate the total volume (m³) for each emplacement by summing up product volumes.

In [15]:
# Debug: Check volume data in produits table
print("=== DEBUGGING VOLUME DATA ===\n")
print(f"Produits columns: {list(produits_df.columns)}")
print(f"\nChecking 'volume pcs (m3)' column:")
print(f"  Data type: {produits_df['volume pcs (m3)'].dtype}")
print(f"  Non-null count: {produits_df['volume pcs (m3)'].notna().sum()} / {len(produits_df)}")
print(f"  Sample values:")
print(produits_df[['id_produit', 'sku', 'volume pcs (m3)']].head(10))

# Try to convert to numeric
test_volume = pd.to_numeric(produits_df['volume pcs (m3)'], errors='coerce')
print(f"\nAfter numeric conversion:")
print(f"  Non-zero values: {(test_volume > 0).sum()}")
print(f"  Sum of all volumes: {test_volume.sum():.3f}")
print(f"  Max volume: {test_volume.max()}")
print(f"  Min non-zero volume: {test_volume[test_volume > 0].min() if (test_volume > 0).any() else 'N/A'}")

=== DEBUGGING VOLUME DATA ===

Produits columns: ['id_produit', 'sku', 'nom_produit', 'unite_mesure', 'categorie', 'actif', 'colisage fardeau', 'colisage palette', 'volume pcs (m3)', 'Poids(kg)', 'Is_Gerbable']

Checking 'volume pcs (m3)' column:
  Data type: object
  Non-null count: 1583 / 1583
  Sample values:
  id_produit    sku volume pcs (m3)
0      texte  texte          nombre
1          O      O               O
2      31334    NaN          0.0002
3      31335    NaN         0.00025
4      31336    NaN         0.00025
5      31337    NaN         0.00025
6      31338    NaN        0.000125
7      31339    NaN         0.00025
8      31340    NaN         0.00025
9      31341    NaN         0.00025

After numeric conversion:
  Non-zero values: 1506
  Sum of all volumes: 48.310
  Max volume: 0.09
  Min non-zero volume: 2e-06


In [19]:
# Calculate total volume for each emplacement
emplacement_volumes = {}

# Convert id_produit to numeric and create volume lookup
produits_df['id_produit_numeric'] = pd.to_numeric(produits_df['id_produit'], errors='coerce')
produits_df['volume_numeric'] = pd.to_numeric(produits_df['volume pcs (m3)'], errors='coerce').fillna(0)

# Create lookup using numeric product IDs
product_volume_lookup = produits_df.set_index('id_produit_numeric')['volume_numeric'].to_dict()

print(f"Product volume lookup created: {len(product_volume_lookup)} products")
print(f"Products with volume > 0: {sum(1 for v in product_volume_lookup.values() if v > 0)}\n")

matches_found = 0
total_volume_calculated = 0.0

for emplacement_id, products in final_inventory.items():
    total_volume = 0.0
    
    for product_id, quantity in products.items():
        # Get volume per piece for this product
        volume_per_piece = product_volume_lookup.get(product_id, 0.0)
        
        if volume_per_piece > 0:
            matches_found += 1
        
        # Calculate total volume for this product
        product_total_volume = quantity * volume_per_piece
        total_volume += product_total_volume
    
    emplacement_volumes[emplacement_id] = round(total_volume, 6)
    total_volume_calculated += total_volume

print(f"✅ Matches found (volume > 0): {matches_found}")
print(f"✓ Calculated volumes for {len(emplacement_volumes)} emplacements")
print(f"\n=== VOLUME STATISTICS ===")
print(f"Total volume across all emplacements: {total_volume_calculated:.3f} m³")
if emplacement_volumes:
    print(f"Average volume per emplacement: {total_volume_calculated / len(emplacement_volumes):.3f} m³")
    print(f"Max volume in an emplacement: {max(emplacement_volumes.values()):.3f} m³")
    print(f"Min volume in an emplacement: {min(emplacement_volumes.values()):.3f} m³")
    print(f"Emplacements with volume > 0: {sum(1 for v in emplacement_volumes.values() if v > 0)}")

Product volume lookup created: 1583 products
Products with volume > 0: 1506

✅ Matches found (volume > 0): 2439
✓ Calculated volumes for 658 emplacements

=== VOLUME STATISTICS ===
Total volume across all emplacements: 988389.005 m³
Average volume per emplacement: 1502.111 m³
Max volume in an emplacement: 107504.014 m³
Min volume in an emplacement: 0.002 m³
Emplacements with volume > 0: 658


In [20]:
# Display sample volumes
print("=== SAMPLE EMPLACEMENT VOLUMES ===\n")

# Sort by volume (descending) and show top 10
sorted_volumes = sorted(emplacement_volumes.items(), key=lambda x: x[1], reverse=True)

print("Top 10 emplacements by volume:")
for i, (emp_id, volume) in enumerate(sorted_volumes[:10], 1):
    products_count = len(final_inventory[emp_id])
    print(f"{i}. Emplacement {emp_id}: {volume:.3f} m³ ({products_count} products)")

print("\n" + "="*50)

=== SAMPLE EMPLACEMENT VOLUMES ===

Top 10 emplacements by volume:
1. Emplacement 214: 107504.014 m³ (143 products)
2. Emplacement 364: 106360.535 m³ (29 products)
3. Emplacement 627: 89730.946 m³ (79 products)
4. Emplacement 1404: 72662.890 m³ (79 products)
5. Emplacement 625: 69448.171 m³ (51 products)
6. Emplacement 626: 55319.551 m³ (42 products)
7. Emplacement 555: 45383.024 m³ (23 products)
8. Emplacement 365: 24958.069 m³ (33 products)
9. Emplacement 1406: 24619.231 m³ (45 products)
10. Emplacement 1403: 18799.416 m³ (87 products)



In [23]:
# Save emplacement volumes to JSON
volume_output_file = OUTPUT_PATH / 'emplacement_volumes.json'

with open(volume_output_file, 'w', encoding='utf-8') as f:
    json.dump(emplacement_volumes, f, indent=2, ensure_ascii=False)

print(f"✓ Emplacement volumes exported to: {volume_output_file}")
print(f"  File size: {volume_output_file.stat().st_size / 1024:.2f} KB")
print(f"\n✅ Volume calculation complete!")

# Display JSON structure sample
print("\n=== JSON STRUCTURE SAMPLE ===")
sample_items = list(emplacement_volumes.items())[:3]
print("{")
for i, (emp_id, volume) in enumerate(sample_items):
    comma = "," if i < len(sample_items) - 1 else ""
    print(f'  "{emp_id}": {volume}{comma}')
if len(emplacement_volumes) > 3:
    print(f"  ... ({len(emplacement_volumes) - 3} more emplacements)")
print("}")

✓ Emplacement volumes exported to: ..\..\data\processed\emplacement_volumes.json
  File size: 11.44 KB

✅ Volume calculation complete!

=== JSON STRUCTURE SAMPLE ===
{
  "54": 41.44,
  "39": 6.66,
  "531": 166.76
  ... (655 more emplacements)
}


## 8. (Bonus) Create Combined Inventory with Volume Data

Create a comprehensive JSON that includes both product inventory and total volume per emplacement.

In [24]:
# Create combined inventory with volume information
combined_inventory = {}

for emplacement_id, products in final_inventory.items():
    combined_inventory[emplacement_id] = {
        'total_volume_m3': emplacement_volumes.get(emplacement_id, 0),
        'product_count': len(products),
        'products': {str(prod_id): qty for prod_id, qty in products.items()}
    }

# Save combined inventory
combined_output_file = OUTPUT_PATH / 'emplacement_inventory_with_volumes.json'
with open(combined_output_file, 'w', encoding='utf-8') as f:
    json.dump(combined_inventory, f, indent=2, ensure_ascii=False)

print(f"✓ Combined inventory exported to: {combined_output_file}")
print(f"  File size: {combined_output_file.stat().st_size / 1024:.2f} KB")

# Display sample structure
print("\n=== COMBINED JSON STRUCTURE SAMPLE ===")
if combined_inventory:
    sample_emp = list(combined_inventory.keys())[0]
    print(f'"{sample_emp}": {{')
    print(f'  "total_volume_m3": {combined_inventory[sample_emp]["total_volume_m3"]},')
    print(f'  "product_count": {combined_inventory[sample_emp]["product_count"]},')
    print(f'  "products": {{')
    sample_products = list(combined_inventory[sample_emp]["products"].items())[:2]
    for i, (prod_id, qty) in enumerate(sample_products):
        comma = "," if i < len(sample_products) - 1 else ""
        print(f'    "{prod_id}": {qty}{comma}')
    if len(combined_inventory[sample_emp]["products"]) > 2:
        print(f'    ... ({len(combined_inventory[sample_emp]["products"]) - 2} more products)')
    print('  }')
    print('}')

print("\n" + "="*60)
print("🎉 ALL PROCESSING COMPLETE!")
print("="*60)
print("\nGenerated files:")
print(f"  1. {OUTPUT_PATH / 'emplacement_inventory.json'} - Simple inventory")
print(f"  2. {OUTPUT_PATH / 'emplacement_inventory_enriched.json'} - Enriched with product details")
print(f"  3. {OUTPUT_PATH / 'emplacement_volumes.json'} - Total volumes per emplacement")
print(f"  4. {OUTPUT_PATH / 'emplacement_inventory_with_volumes.json'} - Combined data")
print("="*60)

✓ Combined inventory exported to: ..\..\data\processed\emplacement_inventory_with_volumes.json
  File size: 116.32 KB

=== COMBINED JSON STRUCTURE SAMPLE ===
"54": {
  "total_volume_m3": 41.44,
  "product_count": 1,
  "products": {
    "31850": 1120
  }
}

🎉 ALL PROCESSING COMPLETE!

Generated files:
  1. ..\..\data\processed\emplacement_inventory.json - Simple inventory
  2. ..\..\data\processed\emplacement_inventory_enriched.json - Enriched with product details
  3. ..\..\data\processed\emplacement_volumes.json - Total volumes per emplacement
  4. ..\..\data\processed\emplacement_inventory_with_volumes.json - Combined data


## 9. Clean JSON - Remove Invalid Emplacement IDs

Remove any emplacement IDs from the JSON that don't exist in the emplacements.csv file.

In [25]:
# Get valid emplacement IDs from CSV
valid_emplacement_ids = set(emplacements_df['id_emplacement'].astype(str).values)

print(f"Valid emplacement IDs from CSV: {len(valid_emplacement_ids)}")
print(f"Current JSON emplacement IDs: {len(combined_inventory)}")

# Find invalid IDs
invalid_ids = [emp_id for emp_id in combined_inventory.keys() if emp_id not in valid_emplacement_ids]

print(f"\n❌ Invalid IDs to remove: {len(invalid_ids)}")
if invalid_ids[:10]:  # Show first 10
    print(f"Sample invalid IDs: {invalid_ids[:10]}")

# Clean all JSON inventories
cleaned_combined_inventory = {emp_id: data for emp_id, data in combined_inventory.items() if emp_id in valid_emplacement_ids}
cleaned_json_inventory = {emp_id: data for emp_id, data in json_inventory.items() if emp_id in valid_emplacement_ids}
cleaned_emplacement_volumes = {emp_id: vol for emp_id, vol in emplacement_volumes.items() if emp_id in valid_emplacement_ids}
cleaned_enriched_inventory = {emp_id: data for emp_id, data in enriched_inventory.items() if emp_id in valid_emplacement_ids}

print(f"\n✓ Cleaned combined inventory: {len(cleaned_combined_inventory)} emplacements")
print(f"✓ Cleaned simple inventory: {len(cleaned_json_inventory)} emplacements")
print(f"✓ Cleaned volumes: {len(cleaned_emplacement_volumes)} emplacements")
print(f"✓ Cleaned enriched inventory: {len(cleaned_enriched_inventory)} emplacements")

Valid emplacement IDs from CSV: 840
Current JSON emplacement IDs: 658

❌ Invalid IDs to remove: 223
Sample invalid IDs: ['54', '39', '531', '366', '628', '627', '284', '529', '653', '524']

✓ Cleaned combined inventory: 435 emplacements
✓ Cleaned simple inventory: 435 emplacements
✓ Cleaned volumes: 435 emplacements
✓ Cleaned enriched inventory: 435 emplacements


In [26]:
# Save cleaned JSON files (overwriting the originals)
with open(OUTPUT_PATH / 'emplacement_inventory.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_json_inventory, f, indent=2, ensure_ascii=False)

with open(OUTPUT_PATH / 'emplacement_inventory_enriched.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_enriched_inventory, f, indent=2, ensure_ascii=False)

with open(OUTPUT_PATH / 'emplacement_volumes.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_emplacement_volumes, f, indent=2, ensure_ascii=False)

with open(OUTPUT_PATH / 'emplacement_inventory_with_volumes.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_combined_inventory, f, indent=2, ensure_ascii=False)

print("✅ All JSON files have been cleaned and saved!")
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Total valid emplacements: {len(cleaned_combined_inventory)}")
print(f"Invalid IDs removed: {len(invalid_ids)}")
print(f"Total volume: {sum(cleaned_emplacement_volumes.values()):.3f} m³")
print(f"\nAll files saved to: {OUTPUT_PATH}")
print("="*60)

✅ All JSON files have been cleaned and saved!

FINAL SUMMARY
Total valid emplacements: 435
Invalid IDs removed: 223
Total volume: 276796.964 m³

All files saved to: ..\..\data\processed


## 10. Add Empty Emplacements

Add emplacements from CSV that don't have any inventory (empty locations).

In [27]:
# Find emplacements in CSV that are not in JSON
current_json_ids = set(cleaned_combined_inventory.keys())
missing_emplacement_ids = [emp_id for emp_id in valid_emplacement_ids if emp_id not in current_json_ids]

print(f"Emplacements in CSV: {len(valid_emplacement_ids)}")
print(f"Emplacements in JSON: {len(current_json_ids)}")
print(f"Missing emplacements (empty): {len(missing_emplacement_ids)}")

# Add empty emplacements to all inventories
for emp_id in missing_emplacement_ids:
    # Add to combined inventory
    cleaned_combined_inventory[emp_id] = {
        'total_volume_m3': 0,
        'product_count': 0,
        'products': {}
    }
    
    # Add to simple inventory
    cleaned_json_inventory[emp_id] = {}
    
    # Add to volumes
    cleaned_emplacement_volumes[emp_id] = 0
    
    # Add to enriched inventory
    cleaned_enriched_inventory[emp_id] = []

print(f"\n✓ Added {len(missing_emplacement_ids)} empty emplacements")
print(f"✓ Total emplacements now: {len(cleaned_combined_inventory)}")

Emplacements in CSV: 840
Emplacements in JSON: 435
Missing emplacements (empty): 405

✓ Added 405 empty emplacements
✓ Total emplacements now: 840


In [28]:
# Save final JSON files with empty emplacements included
with open(OUTPUT_PATH / 'emplacement_inventory.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_json_inventory, f, indent=2, ensure_ascii=False)

with open(OUTPUT_PATH / 'emplacement_inventory_enriched.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_enriched_inventory, f, indent=2, ensure_ascii=False)

with open(OUTPUT_PATH / 'emplacement_volumes.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_emplacement_volumes, f, indent=2, ensure_ascii=False)

with open(OUTPUT_PATH / 'emplacement_inventory_with_volumes.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_combined_inventory, f, indent=2, ensure_ascii=False)

print("✅ All JSON files updated with empty emplacements!")
print("\n" + "="*60)
print("COMPLETE FINAL SUMMARY")
print("="*60)
print(f"Total emplacements in CSV: {len(valid_emplacement_ids)}")
print(f"Total emplacements in JSON: {len(cleaned_combined_inventory)}")
print(f"Emplacements with inventory: {sum(1 for v in cleaned_emplacement_volumes.values() if v > 0)}")
print(f"Empty emplacements: {sum(1 for v in cleaned_emplacement_volumes.values() if v == 0)}")
print(f"Total volume: {sum(cleaned_emplacement_volumes.values()):.3f} m³")
print(f"\nAll files saved to: {OUTPUT_PATH}")
print("="*60)

✅ All JSON files updated with empty emplacements!

COMPLETE FINAL SUMMARY
Total emplacements in CSV: 840
Total emplacements in JSON: 840
Emplacements with inventory: 435
Empty emplacements: 405
Total volume: 276796.964 m³

All files saved to: ..\..\data\processed


In [35]:
# Find and display the emplacement with maximum volume
if cleaned_emplacement_volumes:
    max_volume = max(cleaned_emplacement_volumes.values())
    max_emplacement_id = max(cleaned_emplacement_volumes, key=cleaned_emplacement_volumes.get)
    
    print("\n" + "="*60)
    print("MAXIMUM VOLUME EMPLACEMENT")
    print("="*60)
    print(f"Emplacement ID: {max_emplacement_id}")
    print(f"Maximum Volume: {max_volume:.6f} m³")
    
    # Show product details for this emplacement
    if max_emplacement_id in cleaned_combined_inventory:
        product_count = cleaned_combined_inventory[max_emplacement_id]['product_count']
        print(f"Number of products: {product_count}")
        
        if product_count > 0:
            print("\nProducts in this emplacement:")
            products = cleaned_combined_inventory[max_emplacement_id]['products']
            for prod_id, qty in list(products.items())[:5]:
                print(f"  - Product {prod_id}: {qty} units")
            if len(products) > 5:
                print(f"  ... and {len(products) - 5} more products")
    
    print("="*60)
else:
    print("No volume data available")


MAXIMUM VOLUME EMPLACEMENT
Emplacement ID: 1404
Maximum Volume: 72662.889666 m³
Number of products: 79

Products in this emplacement:
  - Product 31724: 212520 units
  - Product 31461: 41000 units
  - Product 34015: 59500 units
  - Product 44156: 4674 units
  - Product 32089: 600 units
  ... and 74 more products


## 11. Filter by Type: RESERVE Only

Filter the inventory to include only emplacements where `type_emplacement = 'RESERVE'`.

In [32]:
# Filter emplacements by type_emplacement = 'RESERVE'
print("=== FILTERING BY TYPE: RESERVE ===\n")

# Check column name for emplacement type
print(f"Checking emplacements DataFrame columns:")
print(f"Columns: {list(emplacements_df.columns)}\n")

# Get RESERVE emplacement IDs
reserve_emplacements = emplacements_df[emplacements_df['type_emplacement'] == 'RESERVE']['id_emplacement'].astype(str).unique()
reserve_ids_set = set(reserve_emplacements)

print(f"Total RESERVE emplacements in CSV: {len(reserve_ids_set)}")
print(f"Current emplacements in JSON: {len(cleaned_combined_inventory)}")

# Filter all inventory dictionaries to keep only RESERVE type
filtered_combined_inventory = {emp_id: data for emp_id, data in cleaned_combined_inventory.items() if emp_id in reserve_ids_set}
filtered_json_inventory = {emp_id: data for emp_id, data in cleaned_json_inventory.items() if emp_id in reserve_ids_set}
filtered_emplacement_volumes = {emp_id: vol for emp_id, vol in cleaned_emplacement_volumes.items() if emp_id in reserve_ids_set}
filtered_enriched_inventory = {emp_id: data for emp_id, data in cleaned_enriched_inventory.items() if emp_id in reserve_ids_set}

print(f"\n✓ Filtered combined inventory: {len(filtered_combined_inventory)} emplacements")
print(f"✓ Filtered simple inventory: {len(filtered_json_inventory)} emplacements")
print(f"✓ Filtered volumes: {len(filtered_emplacement_volumes)} emplacements")
print(f"✓ Filtered enriched inventory: {len(filtered_enriched_inventory)} emplacements")

# Statistics
reserve_with_inventory = sum(1 for v in filtered_emplacement_volumes.values() if v > 0)
reserve_empty = sum(1 for v in filtered_emplacement_volumes.values() if v == 0)
total_reserve_volume = sum(filtered_emplacement_volumes.values())

print(f"\n=== RESERVE STATISTICS ===")
print(f"Total RESERVE emplacements: {len(filtered_combined_inventory)}")
print(f"  With inventory: {reserve_with_inventory}")
print(f"  Empty: {reserve_empty}")
print(f"Total volume in RESERVE: {total_reserve_volume:.3f} m³")

=== FILTERING BY TYPE: RESERVE ===

Checking emplacements DataFrame columns:
Columns: ['id_emplacement', 'code_emplacement', 'id_entrepot', 'zone', 'type_emplacement', 'actif', 'Volume (m3)', 'Unnamed: 7']

Total RESERVE emplacements in CSV: 178
Current emplacements in JSON: 840

✓ Filtered combined inventory: 178 emplacements
✓ Filtered simple inventory: 178 emplacements
✓ Filtered volumes: 178 emplacements
✓ Filtered enriched inventory: 178 emplacements

=== RESERVE STATISTICS ===
Total RESERVE emplacements: 178
  With inventory: 103
  Empty: 75
Total volume in RESERVE: 179211.060 m³


In [33]:


with open(OUTPUT_PATH / 'emplacement_inventory_with_volumes.json', 'w', encoding='utf-8') as f:
    json.dump(filtered_combined_inventory, f, indent=2, ensure_ascii=False)

print("✅ All JSON files updated with RESERVE emplacements only!")
print("\n" + "="*60)
print("FINAL FILTERED SUMMARY (RESERVE ONLY)")
print("="*60)
print(f"Total RESERVE emplacements: {len(filtered_combined_inventory)}")
print(f"  With inventory: {sum(1 for v in filtered_emplacement_volumes.values() if v > 0)}")
print(f"  Empty: {sum(1 for v in filtered_emplacement_volumes.values() if v == 0)}")
print(f"Total volume in RESERVE: {sum(filtered_emplacement_volumes.values()):.3f} m³")
print(f"\nAll filtered files saved to: {OUTPUT_PATH}")
print("="*60)

✅ All JSON files updated with RESERVE emplacements only!

FINAL FILTERED SUMMARY (RESERVE ONLY)
Total RESERVE emplacements: 178
  With inventory: 103
  Empty: 75
Total volume in RESERVE: 179211.060 m³

All filtered files saved to: ..\..\data\processed


In [34]:
# Display sample of RESERVE emplacements
print("\n=== SAMPLE RESERVE EMPLACEMENTS (TOP 5 BY VOLUME) ===\n")

if filtered_emplacement_volumes:
    # Sort by volume and show top 5
    sorted_reserve = sorted(filtered_emplacement_volumes.items(), key=lambda x: x[1], reverse=True)
    
    for i, (emp_id, volume) in enumerate(sorted_reserve[:5], 1):
        product_count = filtered_combined_inventory[emp_id]['product_count']
        print(f"{i}. Emplacement {emp_id}:")
        print(f"   Volume: {volume:.3f} m³")
        print(f"   Products: {product_count}")
        
        # Show a sample product if available
        if product_count > 0:
            products = filtered_combined_inventory[emp_id]['products']
            first_product = list(products.items())[0]
            print(f"   Sample: Product {first_product[0]} - {first_product[1]} units")
        print()
else:
    print("No RESERVE emplacements found")

print("✅ RESERVE filtering complete!")


=== SAMPLE RESERVE EMPLACEMENTS (TOP 5 BY VOLUME) ===

1. Emplacement 1404:
   Volume: 72662.890 m³
   Products: 79
   Sample: Product 31724 - 212520 units

2. Emplacement 1406:
   Volume: 24619.231 m³
   Products: 45
   Sample: Product 31982 - 876 units

3. Emplacement 1405:
   Volume: 12710.081 m³
   Products: 54
   Sample: Product 34205 - 19624 units

4. Emplacement 1323:
   Volume: 9767.297 m³
   Products: 75
   Sample: Product 32081 - 1296 units

5. Emplacement 1398:
   Volume: 3977.820 m³
   Products: 5
   Sample: Product 35578 - 13100 units

✅ RESERVE filtering complete!


In [37]:
# Load JSON file and find maximum total_volume_m3
print("=== FINDING MAXIMUM VOLUME FROM JSON FILE ===\n")

# Load the JSON file from disk
json_file_path = OUTPUT_PATH / 'emplacement_inventory_with_volumes.json'

print(f"Loading: {json_file_path}")

with open(json_file_path, 'r', encoding='utf-8') as f:
    inventory_data = json.load(f)

print(f"✓ Loaded {len(inventory_data)} emplacements from JSON\n")

# Extract all volumes and find maximum
max_volume = 0
max_emp_id = None

for emp_id, data in inventory_data.items():
    volume = data['total_volume_m3']
    if volume > max_volume:
        max_volume = volume
        max_emp_id = emp_id

# Display results
print("="*60)
print("MAXIMUM VOLUME EMPLACEMENT")
print("="*60)
print(f"Emplacement ID: {max_emp_id}")
print(f"Maximum Volume: {max_volume:.6f} m³")
print(f"Product Count: {inventory_data[max_emp_id]['product_count']}")

# Show products in this emplacement
if inventory_data[max_emp_id]['product_count'] > 0:
    print(f"\nProducts in this emplacement:")
    products = inventory_data[max_emp_id]['products']
    product_list = list(products.items())[:5]
    for i, (prod_id, qty) in enumerate(product_list, 1):
        print(f"  {i}. Product {prod_id}: {qty} units")
    if len(products) > 5:
        print(f"  ... and {len(products) - 5} more products")

# Calculate statistics
all_volumes = [data['total_volume_m3'] for data in inventory_data.values()]
print(f"\n=== VOLUME STATISTICS ===")
print(f"Total emplacements: {len(inventory_data)}")
print(f"Minimum volume: {min(all_volumes):.6f} m³")
print(f"Average volume: {sum(all_volumes) / len(all_volumes):.6f} m³")
print(f"Maximum volume: {max_volume:.6f} m³")
print(f"Emplacements with volume > 0: {sum(1 for v in all_volumes if v > 0)}")
print("="*60)

=== FINDING MAXIMUM VOLUME FROM JSON FILE ===

Loading: ..\..\data\processed\emplacement_inventory_with_volumes.json
✓ Loaded 175 emplacements from JSON

MAXIMUM VOLUME EMPLACEMENT
Emplacement ID: 1323
Maximum Volume: 9767.296921 m³
Product Count: 75

Products in this emplacement:
  1. Product 32081: 1296 units
  2. Product 31343: 1776 units
  3. Product 32089: 1863 units
  4. Product 31463: 3100 units
  5. Product 32086: 3204 units
  ... and 70 more products

=== VOLUME STATISTICS ===
Total emplacements: 175
Minimum volume: 0.000000 m³
Average volume: 395.536327 m³
Maximum volume: 9767.296921 m³
Emplacements with volume > 0: 100


In [40]:
# Calculate available slots (assume max capacity = 10,000 m³ per emplacement)
print("=== CALCULATING AVAILABLE SLOTS ===\n")

MAX_CAPACITY_M3 = 10000  # Maximum capacity per emplacement in m³

# Load the current inventory JSON
json_file_path = OUTPUT_PATH / 'emplacement_inventory_with_volumes.json'

with open(json_file_path, 'r', encoding='utf-8') as f:
    inventory_data = json.load(f)

print(f"✓ Loaded {len(inventory_data)} emplacements")
print(f"Maximum capacity per emplacement: {MAX_CAPACITY_M3:,.0f} m³\n")

# Calculate available slots for each emplacement
available_slots = {}

for emp_id, data in inventory_data.items():
    current_volume = data['total_volume_m3']
    available_space = MAX_CAPACITY_M3 - current_volume
    available_slots[emp_id] = round(available_space, 6)

# Save to new JSON file
available_slots_file = OUTPUT_PATH / 'emplacement_available_slots.json'

with open(available_slots_file, 'w', encoding='utf-8') as f:
    json.dump(available_slots, f, indent=2, ensure_ascii=False)

print(f"✓ Available slots JSON created: {available_slots_file}")
print(f"  File size: {available_slots_file.stat().st_size / 1024:.2f} KB")

# Statistics
total_used = sum(data['total_volume_m3'] for data in inventory_data.values())
total_capacity = MAX_CAPACITY_M3 * len(inventory_data)
total_available = sum(available_slots.values())
utilization_rate = (total_used / total_capacity) * 100

print(f"\n=== CAPACITY STATISTICS ===")
print(f"Total emplacements: {len(inventory_data)}")
print(f"Total capacity: {total_capacity:,.2f} m³")
print(f"Total used: {total_used:,.2f} m³")
print(f"Total available: {total_available:,.2f} m³")
print(f"Utilization rate: {utilization_rate:.2f}%")

# Show emplacements with most available space
print(f"\n=== TOP 5 EMPLACEMENTS BY AVAILABLE SPACE ===")
sorted_available = sorted(available_slots.items(), key=lambda x: x[1], reverse=True)
for i, (emp_id, available) in enumerate(sorted_available[:5], 1):
    current = inventory_data[emp_id]['total_volume_m3']
    print(f"{i}. Emplacement {emp_id}: {available:,.2f} m³ available (current: {current:,.2f} m³)")

print("="*60)

=== CALCULATING AVAILABLE SLOTS ===

✓ Loaded 175 emplacements
Maximum capacity per emplacement: 10,000 m³

✓ Available slots JSON created: ..\..\data\processed\emplacement_available_slots.json
  File size: 3.21 KB

=== CAPACITY STATISTICS ===
Total emplacements: 175
Total capacity: 1,750,000.00 m³
Total used: 69,218.86 m³
Total available: 1,680,781.14 m³
Utilization rate: 3.96%

=== TOP 5 EMPLACEMENTS BY AVAILABLE SPACE ===
1. Emplacement 1395: 10,000.00 m³ available (current: 0.00 m³)
2. Emplacement 1344: 10,000.00 m³ available (current: 0.00 m³)
3. Emplacement 416: 10,000.00 m³ available (current: 0.00 m³)
4. Emplacement 891: 10,000.00 m³ available (current: 0.00 m³)
5. Emplacement 410: 10,000.00 m³ available (current: 0.00 m³)


In [41]:
import numpy as np

# Create a 30x44 matrix of zeros
matrix = np.zeros((30, 44))

print(matrix)
print(f"\nShape: {matrix.shape}")

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

Shape: (30, 44)
